# VisionDrive — unchanged pretrained YOLO11n evaluation

## 1. Research objective

**Primary evaluation: PERSON only.** Pilot figure QA exposed the Open Images Stool/COCO chair taxonomy mismatch. The two-class pilot is exploratory. A fresh **400-image** person-only cohort excludes all previously inspected and baseline images. Its source-only selection, mapping and thresholds were frozen before fresh-cohort inference.

Measure original COCO-pretrained YOLO11n on real indoor images. **No training or fine-tuning occurs.** Results address person localization, not all obstacles, calibrated distance or physical collision avoidance.

Public data is external to this project but is **not proven independent of COCO pretraining**. No team camera recordings were supplied. The older ESPDet experiment remains separate in `train_bottle_cup_box.ipynb`.

Run all cells using the evaluation environment from the repository README. Downloads are cached, checked and explicit. Expect original weights in `weights/laptop/yolo11n.pt`; run the setup script first. Outputs go to `outputs/evaluation/yolo11n/`.

## 2. Environment setup
The device is explicit. Use `DEVICE="cpu"` only when intentionally measuring CPU performance; never silently substitute it for CUDA.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT=Path.cwd().resolve()
if not (ROOT/'demo.py').exists() and (ROOT.parent/'demo.py').exists(): ROOT=ROOT.parent
assert (ROOT/'demo.py').exists(), "Run from repository root or notebooks/"
sys.path.insert(0,str(ROOT))
os.environ['YOLO_CONFIG_DIR']=str(ROOT/'.cache/ultralytics')
os.environ['YOLO_AUTOINSTALL']='false'
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
import torch, ultralytics, numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from IPython.display import display, Image, Markdown
from visiondrive.evaluation.data import acquire, validate, read_jsonl, digest, NAMES, COCO_NAMES, write_json
from visiondrive.evaluation.runner import load_config, environment, load_model, native_dataset, extract_native, predictions_and_benchmark, assemble_results, save_results
from visiondrive.evaluation.metrics import aggregate, bootstrap, conditions, match
from visiondrive.evaluation.report import exploration
DEVICE='0'
config=load_config(device=DEVICE)
dataset=ROOT/config['dataset']; output=ROOT/config['output']
output.mkdir(parents=True,exist_ok=True)
display(pd.Series(environment(config),name='environment'))

## 3. Dataset acquisition and validation
Restore the frozen per-image manifest from official Open Images public storage. Every photo records author, original page, download URL, CC BY 2.0 license, dimensions, SHA-256 and source split. Original annotations are CC BY 4.0.

The source selection uses Building-family `IsInside=1`, human-verified **person presence or absence**, and excludes target group-of, depiction, inside/unknown flags, label conflicts and ambiguous person hierarchy overlaps. No missing label is assumed negative. Person subtypes are propagated; coincident cross-source boxes are collapsed, while remaining ambiguous overlaps exclude the image. Source Chair is not complete for COCO chair because Stool is separate, so chair is not scored.

The frozen cohort contains **400 images (80 dev / 320 test)** and **923 person instances (191 dev / 732 test)**. Each partition is 75% person-positive and 25% verified negative, with one photo per photographer and no photographer shared across splits. Development/test preserve official source assignments. All earlier inspected cohort and baseline IDs are excluded. Source-only sampling used seed 20261017 before inference. This satisfies the initial 300–500 practical target without claiming it is a statistical minimum. SHA-256 and 64-bit dHash checks reject exact/likely near duplicates; photographer identity is a scene proxy, not proof of physical scene independence.

See `docs/dataset_card.md` for candidate comparisons, licensing, source CSV checksums, exclusion counts and pretraining uncertainty. No training split is created for unchanged YOLO; a required Ultralytics `train` YAML key aliases development and is never used.

In [ ]:
records=acquire(dataset)
validation=validate(dataset)
assert validation['valid'], validation['errors']
assert validation['manifest_sha256']==config['dataset_manifest_sha256']
assert digest(dataset/'manifest.jsonl')==digest(ROOT/'data/public/indoor-evaluation-manifest.jsonl')
display(validation)
write_json(output/'dataset_validation.json',validation)

## 4. Dataset exploration
Counts are objects and images separately. Ground-truth sheets allow visual inspection; source protocol and structural checks do not constitute independent exhaustive reannotation.

In [ ]:
from collections import Counter
print('Images:',len(records),' | Source splits:',dict(Counter(r['source']['original_split'] for r in records)))
distribution=pd.DataFrame([{'split':s,'class':name,'images':sum(r['split']==s and any(o['label']==name for o in r['objects']) for r in records),
                            'instances':sum(o['label']==name for r in records if r['split']==s for o in r['objects'])}
                           for s in ['dev','test'] for name in config['class_names']])
display(distribution)
display(pd.Series(validation['warnings'],name='limitations'))
exploration(dataset,output)
display(Image(filename=str(output/'class_distribution.png')))
display(Image(filename=str(output/'ground_truth_review/sheet_01.jpg'),width=1100))

## 5. Load the original pretrained model
The original checkpoint and all 80 class names are checked. Only COCO ID 0 (person) is primary-scored; chair is excluded due to the Stool taxonomy mismatch; the live application retains all 80 categories.

In [ ]:
initial_sha=digest(ROOT/config['weights'])
assert initial_sha==config['weights_sha256']
model=load_model(config)
display({'checkpoint':config['weights'],'sha256':initial_sha,'parameters_before_fusion':sum(p.numel() for p in model.model.parameters()),
         'device':config['device'],'names':model.names})
assert model.names[0]=='person' and model.names[56]=='chair' and len(model.names)==80

## 6. Run native evaluation and genuine predictions
The dataset YAML preserves **all 80 original IDs**, and the filter uses `[0]`, not custom IDs `[0,1]`. Image lists come from the frozen manifest so excluded cached images cannot enter evaluation.

Native `model.val()` integrates AP over confidence at IoU .50:.05:.95. The displayed native P/R are its internally selected operating point and are **not** our fixed runtime P/R. Operational P/R/F1 below use .35 confidence and .50 IoU. NMS IoU=.70, max detections=300, square letterboxing to 640, FP32, batch=1. These settings were fixed before final test inference.

In [ ]:
dataset_yaml=native_dataset(dataset,output)
native_metrics=model.val(data=str(dataset_yaml),split='test',imgsz=config['image_size'],batch=config['batch'],
    device=config['device'],half=config['half'],classes=config['class_ids'],
    conf=config['ap_confidence_floor'],iou=config['nms_iou'],max_det=config['max_detections'],
    plots=True,workers=0,rect=False,verbose=False,save_json=False,project=str(output),name='native',exist_ok=True)
native,curves=extract_native(native_metrics,config)
test_records=[r for r in records if r['split']=='test']
rows,benchmark=predictions_and_benchmark(model,config,dataset,test_records)
assert len(rows)==len(test_records)
assert digest(ROOT/config['weights'])==initial_sha

## 7. Detection metrics and statistical uncertainty
Predictions are confidence ordered, matched one-to-one to the highest-IoU available same-class ground truth at IoU≥.50. Duplicate predictions are false positives. Undefined ratios are assigned zero. Micro metrics pool TP/FP/FN. Macro F1 is the arithmetic mean of per-class F1. It is not the harmonic mean of macro precision and recall.

mAP@75 is the sixth threshold in `.50,.55,.60,.65,.70,.75,.80,.85,.90,.95`. Native AP matching differs from the fixed-point greedy matcher. The two protocols are reported separately. mAP is not classification accuracy.

95% percentile confidence intervals resample photographer groups 1,000 times. They do not resolve unknown physical scene relationships, sampling bias, or pretraining overlap; AP confidence intervals are not claimed.

In [ ]:
fixed=aggregate(rows,config['confidence'],config['matching_iou'],config['class_ids'])
micro=fixed['micro']
assert micro['tp']+micro['fn']==sum(o['class_id'] in config['class_ids'] for r in rows for o in r['objects'])
assert np.isclose(micro['f1'],2*micro['tp']/(2*micro['tp']+micro['fp']+micro['fn']))
assert np.isclose(fixed['macro']['f1'],np.mean([v['f1'] for v in fixed['per_class'].values()]))
result=assemble_results(model,config,validation,native,rows,benchmark,initial_sha)
display(pd.Series({**micro,**{k:native[k] for k in ['map50','map75','map50_95']},'macro_f1':fixed['macro']['f1']}))
display(result['bootstrap'])

## 8. Per-class results
Only person is primary-scored. Pilot chair scoring was excluded due to a Stool/Chair taxonomy mismatch. The final fresh person-only protocol was frozen before inference. Earlier chair numbers are exploratory, not reliable furniture performance.

In [ ]:
per_class=pd.DataFrame(result['per_class']).T
display(per_class[['ground_truth','tp','fp','fn','precision','recall','f1','ap50','ap75','ap50_95']])
print('Other 79 COCO classes are not primary-scored, not scored as absent or failed.')

## 9. Diagnostic plots and prediction evidence
The fixed-point confusion matrices show same-class TP on the diagonal, misses in the final column and unmatched predictions in the final row. A wrong class contributes FP+FN; off-diagonal class substitution counts are not inferred. Normalization is by row, including the unmatched-prediction row. Native plots are retained separately under `native/`.

Example selection is transparent: three images with most misses, three with most false positives, and three deterministic initial examples. They illustrate observed failure modes and are not used to tune the model.

In [ ]:
save_results(result,rows,curves,dataset,output)
for name in ['confusion_matrix.png','confusion_matrix_normalized.png','precision_recall_curve.png','f1_confidence_curve.png','per_class_performance.png']:
    display(Image(filename=str(output/name),width=750))
for name in ['missed_1.jpg','false_positive_1.jpg','example_1.jpg']:
    display(Image(filename=str(output/'prediction_examples'/name),width=1000))

## 10. Inference benchmarking
The prediction pass in Section 6 includes 10 warmups and synchronizes CUDA before and after each measured call. The first 100 test images (manifest order) provide latency samples; disk decode is outside timing. Ultralytics reports synchronized pre/network/post phases; external wall time includes Python overhead. Mean, median, population SD, p95, and 1000/mean FPS are recorded. Batch=1, FP32, 640×640.

**Model-only FPS excludes depth, camera, GUI, controller, UDP and motors.** Full perception/controller/render benchmarking is separate in `scripts/benchmark_system.py`; no physical navigation result follows from either speed measure.

In [ ]:
timing=pd.DataFrame({phase:values for phase,values in benchmark.items() if isinstance(values,dict)}).T
display(timing)
display({k:v for k,v in benchmark.items() if not isinstance(v,dict)})

## 11. Evidence-based error analysis
Source occlusion/truncation flags, source-pixel object area, image luminance and number of target instances provide observational condition recall. Small means area<32² source pixels, medium<96², large≥96². Dark means full-frame grayscale mean<70; many-target means ≥5 target objects. These bins are confounded and are not causal lighting or clutter experiments. Perspective and background complexity have no independent labels, so we do not invent their numerical effects.

In [ ]:
condition_table=pd.DataFrame(result['condition_recall'])
display(condition_table)
misses=sorted([(sum(match(r,cid,config['confidence'],config['matching_iou'])['fn'] for cid in config['class_ids']),r['image_id']) for r in rows],reverse=True)
display(pd.DataFrame(misses[:5],columns=['false_negatives','source_image_id']))
print('Use the exported source-attributed examples to discuss specific misses; do not infer unmeasured physical distance.')

## 12. Research interpretation and reproducibility record
The report below is generated from the measured predictions. Dataset/model/configuration/environment/source digests accompany it. No published benchmark is substituted for local measurements. Public-data results do not establish performance on our future car camera or prove independence from pretraining.

Depth metrics are pending valid aligned metric ground truth. Future camera data must keep complete recording sessions together, annotate every scored category (including verified empty images), use independent acceptable movement-action sets, and remain separate from public evaluation. See `docs/camera-evaluation.md`. All existing STOP/watchdog protections remain in the application.

In [ ]:
assert result['weights_unchanged'] and digest(ROOT/config['weights'])==config['weights_sha256']
display(Markdown((output/'evaluation_report.md').read_text(encoding='utf-8')))
print('All outputs:',output.relative_to(ROOT).as_posix())
print('Checkpoint unchanged; no training executed.')